# S23 · Roll the dice to estimate pi and an area

When the maths is hard, industry does something cheeky: it stops solving and starts
rolling the dice. Here we estimate the number pi by throwing thousands of random
darts at a square, then estimate the area under a curve the same way. Both work for
one reason: the **law of large numbers**. The more darts we throw, the closer our
answer gets.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on each
  cell, top to bottom, and read the plain-English note above each one.
- New to "roll the dice and average"? Open the primer
  `primers/the_law_of_large_numbers.md` for a ten-minute, picture-first version.
- Already confident with code or with probability? Run the simple version, then look
  for the cell marked **Stretch (optional)** near the end (importance sampling).
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook only uses numpy and matplotlib, which Colab already ships.
# So there is nothing to install here.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                 # random numbers and fast maths
import matplotlib.pyplot as plt      # drawing charts

# Set a seed so everyone gets the same "random" darts and the same answer.
np.random.seed(0)

print("Ready to roll the dice.")

## Step 1 — the idea: throw darts at a square

Picture a square that runs from -1 to 1 in both directions. Inside it sits a circle
of radius 1. Now throw darts so they land at completely random spots in the square.
The fraction that land inside the circle tells us about pi.

Why? The square has area `2 * 2 = 4`. The circle has area `pi * 1^2 = pi`. So the
chance a random dart lands inside the circle is `pi / 4`. Turn that around:
`pi = 4 * (fraction of darts inside)`. We never do any calculus; we just count.

In [ ]:
# How many darts to throw.
number_of_darts = 3000

# Each dart needs a random x and a random y, each between -1 and 1.
dart_x = np.random.uniform(-1, 1, size=number_of_darts)
dart_y = np.random.uniform(-1, 1, size=number_of_darts)

# A dart is inside the circle when its distance from the centre is <= 1.
# Distance squared is x*x + y*y, and the radius is 1, so we compare to 1.
distance_squared = dart_x ** 2 + dart_y ** 2
is_inside = distance_squared <= 1

# Count how many landed inside.
number_inside = np.sum(is_inside)
print("darts thrown :", number_of_darts)
print("darts inside :", number_inside)

## Step 2 — turn the fraction into an estimate of pi

The fraction of darts inside should be close to `pi / 4`, so we multiply it by 4.

In [ ]:
fraction_inside = number_inside / number_of_darts
pi_estimate = 4 * fraction_inside

print("fraction inside     :", round(fraction_inside, 4))
print("our estimate of pi  :", round(pi_estimate, 4))
print("the real value of pi:", round(np.pi, 4))

## Step 3 — draw the darts

Let us see it. Darts inside the circle are coloured blue, darts outside are red. We
draw the circle on top so we can check our colouring.

In [ ]:
plt.figure(figsize=(6, 6))

# Blue points: the darts that landed inside the circle.
plt.scatter(dart_x[is_inside], dart_y[is_inside], s=6, color="#2E75B6", label="inside")

# Red points: the darts that landed outside the circle.
plt.scatter(dart_x[~is_inside], dart_y[~is_inside], s=6, color="#C0392B", label="outside")

# Draw the circle itself so we can see the boundary.
angle = np.linspace(0, 2 * np.pi, 200)
plt.plot(np.cos(angle), np.sin(angle), color="black", linewidth=2)

plt.xlabel("x")
plt.ylabel("y")
plt.title("Throwing random darts to estimate pi")
plt.legend()
plt.gca().set_aspect("equal")   # keep the circle round, not squashed
plt.show()

## Step 4 — watch the estimate settle as we throw more darts

This is the **law of large numbers** in action, the heart of today's session. With
only a few darts the estimate jumps around wildly. As we throw more and more, the
running estimate calms down and hugs the true value of pi.

We compute a *running* estimate: after each dart, we recompute pi using every dart
thrown so far.

In [ ]:
# A fresh, bigger batch of darts so we can watch the estimate over time.
number_of_darts = 20000
dart_x = np.random.uniform(-1, 1, size=number_of_darts)
dart_y = np.random.uniform(-1, 1, size=number_of_darts)
is_inside = (dart_x ** 2 + dart_y ** 2) <= 1

# np.cumsum gives the running total of "inside" darts after each throw.
running_inside_total = np.cumsum(is_inside)

# The dart numbers 1, 2, 3, ... up to the total.
dart_number = np.arange(1, number_of_darts + 1)

# Running estimate of pi after each dart.
running_pi_estimate = 4 * running_inside_total / dart_number

print("estimate after    10 darts:", round(running_pi_estimate[9], 4))
print("estimate after 20000 darts:", round(running_pi_estimate[-1], 4))

## Step 5 — plot the running estimate

The wobbling line is our estimate. The dashed line is the true pi. Notice how the
wobble shrinks as we move right, that is, as more darts pile up.

In [ ]:
plt.figure(figsize=(8, 5))

# Our running estimate after each dart.
plt.plot(dart_number, running_pi_estimate, color="#2E75B6", label="our running estimate")

# The true value of pi, drawn as a flat dashed line for comparison.
plt.axhline(np.pi, color="#C0392B", linestyle="--", linewidth=2, label="true pi")

plt.xlabel("number of darts thrown")
plt.ylabel("estimate of pi")
plt.title("The estimate settles down as we throw more darts")
plt.ylim(2.6, 3.7)
plt.legend()
plt.show()

## Step 6 — the catch: the error shrinks only like 1 / sqrt(N)

More darts means a better estimate, but with sharply diminishing returns. The size
of the error shrinks like `1 / sqrt(N)`, where `N` is the number of darts. In plain
words: to make the error 10 times smaller, you need about 100 times as many darts.
This is why rolling the dice is a lovely idea but an expensive one.

We check it by estimating pi many times for each sample size and measuring how spread
out the answers are. A wider spread means a bigger typical error.

In [ ]:
# The sample sizes we will test.
sample_sizes = [100, 400, 1600, 6400, 25600]

# For each sample size, estimate pi 50 times and record the spread (the error).
error_for_each_size = []

for size in sample_sizes:
    estimates = []
    # Repeat the experiment 50 times to see how much the answer wobbles.
    for repeat in range(50):
        x = np.random.uniform(-1, 1, size=size)
        y = np.random.uniform(-1, 1, size=size)
        inside = (x ** 2 + y ** 2) <= 1
        estimates.append(4 * np.mean(inside))
    # The standard deviation of the estimates measures the typical error.
    error_for_each_size.append(np.std(estimates))

for size, error in zip(sample_sizes, error_for_each_size):
    print("samples:", size, "  typical error:", round(error, 4))

## Step 7 — plot the error on a log-log scale

On a log-log plot a `1 / sqrt(N)` relationship looks like a straight line going
downhill. Our measured errors should sit close to that reference line, confirming the
rule.

In [ ]:
sample_sizes = np.array(sample_sizes)
error_for_each_size = np.array(error_for_each_size)

# A reference line that falls exactly like 1 / sqrt(N), lined up with our first point.
reference_line = error_for_each_size[0] * np.sqrt(sample_sizes[0]) / np.sqrt(sample_sizes)

plt.figure(figsize=(8, 5))
plt.loglog(sample_sizes, error_for_each_size, "o-", color="#2E75B6", label="measured error")
plt.loglog(sample_sizes, reference_line, "--", color="#C0392B", label="1 / sqrt(N) reference")
plt.xlabel("number of samples N")
plt.ylabel("typical error")
plt.title("Error shrinks like 1 / sqrt(N)")
plt.legend()
plt.show()

## Step 8 — the same idea estimates an integral

An integral is just the area under a curve. We can estimate an area the same way:
scatter random points in a box, and the fraction that land *under the curve* is the
fraction of the box that is area.

We will estimate the integral of `f(x) = x^2` from 0 to 1. By hand the answer is
`1/3`, so we have something honest to check against.

In [ ]:
def f(x):
    # The curve whose area under it we want to find.
    return x ** 2

# Throw random points into the box: x between 0 and 1, y between 0 and 1.
number_of_points = 20000
point_x = np.random.uniform(0, 1, size=number_of_points)
point_y = np.random.uniform(0, 1, size=number_of_points)

# A point is "under the curve" when its height y is below f(x).
is_under_curve = point_y <= f(point_x)

# The box has area 1 (1 wide, 1 tall), so the area under the curve is just the
# fraction of points that landed under it.
area_estimate = np.mean(is_under_curve)

print("our estimate of the integral:", round(area_estimate, 4))
print("the exact answer (1/3)      :", round(1 / 3, 4))

## Step 9 — draw the integral picture

Points under the curve are green; points above are grey. The green fraction is our
estimate of the area.

In [ ]:
plt.figure(figsize=(7, 5))

# The two groups of points.
plt.scatter(point_x[is_under_curve], point_y[is_under_curve], s=4, color="#27AE60", label="under curve")
plt.scatter(point_x[~is_under_curve], point_y[~is_under_curve], s=4, color="#95A5A6", label="above curve")

# The curve f(x) = x^2 drawn on top.
curve_x = np.linspace(0, 1, 200)
plt.plot(curve_x, f(curve_x), color="black", linewidth=2.5, label="f(x) = x^2")

plt.xlabel("x")
plt.ylabel("y")
plt.title("Estimating an integral by counting points under the curve")
plt.legend()
plt.show()

### Stretch (optional) — roll the dice more cleverly (importance sampling)

Skip this if you are new to code or probability. If you are comfortable, here is the
professional's answer to the `1/sqrt(N)` tax: instead of throwing darts blindly,
throw more of them where they matter, then reweight so the answer stays correct. The
trick is called **importance sampling**, and it can cut the work by a huge factor.

The demo: estimate the chance that a standard bell-curve value lands above 3. This is
a rare event (about 0.00135), so plain darts almost all miss it and waste effort.
Importance sampling instead samples from a bell curve *shifted* up to 3, where the
event is common, then corrects each sample by the ratio of the true density to the
one we sampled from.

In [ ]:
# Plain Monte Carlo: draw ordinary standard-normal numbers and count how many
# exceed 3. Almost none do, so this estimate is noisy and wasteful.
np.random.seed(0)
sample_count = 20000
plain_samples = np.random.normal(0, 1, size=sample_count)
plain_estimate = np.mean(plain_samples > 3)

# Importance sampling: draw from a bell curve centred at 3 instead, where landing
# above 3 is common. Then reweight each sample so the answer stays correct.
shifted_samples = np.random.normal(3, 1, size=sample_count)

# The reweight factor is (true density) / (density we actually sampled from).
# Both are bell-curve densities, which we write out by hand.
def normal_density(value, centre):
    return np.exp(-0.5 * (value - centre) ** 2) / np.sqrt(2 * np.pi)

true_density = normal_density(shifted_samples, 0)       # the real standard normal
proposal_density = normal_density(shifted_samples, 3)   # the one we sampled from
reweight = true_density / proposal_density

exceeds_three = shifted_samples > 3
importance_estimate = np.mean(reweight * exceeds_three)

true_probability = 0.001349898   # the exact answer, for comparison

print("plain Monte Carlo estimate  :", round(plain_estimate, 6))
print("importance sampling estimate:", round(importance_estimate, 6))
print("true probability            :", round(true_probability, 6))
print()
print("Importance sampling spent its darts where the event actually happens,")
print("so it nails the rare probability that plain darts can barely see.")

## What you just did

You estimated pi and an integral with nothing but random numbers and averaging. Both
work because of the **law of large numbers**: the average of many random samples gets
close to the true value. You also felt the price: the error only shrinks like
`1 / sqrt(N)`, so more accuracy costs a lot more darts, and clever sampling can buy
some of it back.

Next notebook: `02_five_useful_distributions.ipynb`, where we meet the five kinds of
random number that power almost every simulation.